# Churn Prediction

In [ ]:
!pip install pyswarms -q
!pip install imbalanced-learn optuna -q

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR   = '/content/drive/MyDrive/hbp_results2'
RESULTS_DIR = DRIVE_DIR
MODELS_DIR  = f'{DRIVE_DIR}/models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
print('Hasil akan disimpan di:', RESULTS_DIR)

# import glob, os
# for f in glob.glob(f'{RESULTS_DIR}/K04_C*checkpoint.json') + glob.glob(f'{RESULTS_DIR}/K03_C*checkpoint.json'):
#     os.remove(f); print('hapus', f)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('Install selesai')

### Download Dataset

In [ ]:
kaggle_dir = '/root/.kaggle'
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
    drive_kaggle = '/content/drive/MyDrive/kaggle.json'
    if os.path.exists(drive_kaggle):
        import shutil
        shutil.copy(drive_kaggle, f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
        print('kaggle.json disalin dari Drive')
    else:
        print('Upload kaggle.json:')
        from google.colab import files
        uploaded = files.upload()
        with open(f'{kaggle_dir}/kaggle.json', 'wb') as f:
            f.write(list(uploaded.values())[0])
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
        print('kaggle.json berhasil diupload')
else:
    print('kaggle.json sudah ada')


In [ ]:
import kagglehub

# Download K03 — Churn Modelling
k03_dir  = kagglehub.dataset_download('shrutimechlearn/churn-modelling')
k03_path = os.path.join(k03_dir, 'Churn_Modelling.csv')
print('K03 path:', k03_path)

# Download K04 — Telco Customer Churn
k04_dir  = kagglehub.dataset_download('blastchar/telco-customer-churn')
k04_path = os.path.join(k04_dir, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('K04 path:', k04_path)

Path to dataset files: /kaggle/input/datasets/shrutimechlearn/churn-modelling/Churn_Modelling.csv


In [ ]:
k03_drop = ['RowNumber','CustomerId','Surname','Exited','label']
#k04_drop = ['customerID', 'Churn']
k04_drop = ['customerID']

DATASETS = {
    'K03': {
        'name'        : 'Churn Modelling (Perbankan)',
        'path'        : k03_path,
        'target'      : 'Exited',
        'drop_cols'   : k03_drop,
        'positive_val': None,
        'hpo_max_rows': None,
    },
    'K04': {
        'name'        : 'Telco Customer Churn (Telekomunikasi)',
        'path'        : k04_path,
        'target'      : 'Churn',
        #'target'      : 'label',
        'drop_cols'   : k04_drop,
        'positive_val': 'Yes',
        'hpo_max_rows': None,
    },
}

## Exploratory Data Analysis

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

EDA_DIR = f'{RESULTS_DIR}/eda'
os.makedirs(EDA_DIR, exist_ok=True)

In [9]:
# ── Palette ────────────────────────────────────────────────────────────────
C_NO   = '#378ADD'   # blue  = no churn
C_YES  = '#E24B4A'   # red   = churn
C_TEAL = '#1D9E75'
C_AMB  = '#BA7517'
CMAP   = [C_NO, C_YES]

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'figure.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})

In [10]:
k03 = pd.read_csv(k03_path)
k04 = pd.read_csv(k04_path)

In [11]:
k03.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [12]:
k04.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [13]:
# K03: target = Exited (0/1)
k03['churn'] = k03['Exited'] if 'Exited' in k03.columns else k03['label']

# K04: target = Churn (Yes/No) stored as 'label'
k04['TotalCharges'] = pd.to_numeric(
    k04['TotalCharges'],
    errors='coerce'
)

k04['TotalCharges'] = k04['TotalCharges'].fillna(
    k04['TotalCharges'].median()
)

k04['churn'] = (
    (k04['Churn'] == 'Yes').astype(int)
    if k04['Churn'].dtype == object
    else k04['Churn']
)

In [14]:
print(f"K03 loaded: {k03.shape} | churn rate: {k03['churn'].mean()*100:.1f}%")
print(f"K04 loaded: {k04.shape} | churn rate: {k04['churn'].mean()*100:.1f}%")

K03 loaded: (10000, 15) | churn rate: 20.4%
K04 loaded: (7043, 22) | churn rate: 26.5%


In [ ]:
#  FIG 1: DATASET OVERVIEW 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Fig 1 — Class Distribution: K03 vs K04', fontsize=14, fontweight='bold', y=1.01)

for ax, df, name, ir in zip(axes,
                              [k03, k04],
                              ['K03: Churn Modelling (Banking)', 'K04: Telco Customer Churn'],
                              [3.91, 2.77]):
    counts = df['churn'].value_counts().sort_index()
    bars = ax.bar(['No Churn', 'Churn'], counts.values, color=CMAP, width=0.5, edgecolor='white')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts.values)*1.25)
    ax.text(0.98, 0.97, f'IR = {ir:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF3CD', edgecolor='#BA7517', alpha=0.9))

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig1_class_distribution.png')
plt.close()
print('Saved: fig1_class_distribution.png')

#  FIG 2: CHURN BY CATEGORICAL FEATURES
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Fig 2 — K03: Churn Rate by Categorical Features', fontsize=14, fontweight='bold')

def churn_bar(ax, df, col, title, top_n=None, rotate=False):
    tbl = df.groupby(col)['churn'].mean().sort_values(ascending=False)*100
    if top_n: tbl = tbl.head(top_n)
    colors = [C_YES if v > 25 else C_AMB if v > 15 else C_TEAL for v in tbl.values]
    bars = ax.bar(tbl.index.astype(str), tbl.values, color=colors, edgecolor='white')
    for bar, val in zip(bars, tbl.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, tbl.max()*1.3)
    if rotate: plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.axhline(df['churn'].mean()*100, color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'Avg {df["churn"].mean()*100:.1f}%')
    ax.legend(fontsize=8)

churn_bar(axes[0,0], k03, 'Geography',       'By Geography')
churn_bar(axes[0,1], k03, 'Gender',           'By Gender')
churn_bar(axes[0,2], k03, 'NumOfProducts',    'By Num of Products')
churn_bar(axes[1,0], k03, 'IsActiveMember',   'By Active Member (0=No, 1=Yes)')
churn_bar(axes[1,1], k03, 'HasCrCard',        'By Has Credit Card (0=No, 1=Yes)')

# Age group
k03['AgeGroup'] = pd.cut(k03['Age'], bins=[0,30,40,50,60,100], labels=['<30','30-40','40-50','50-60','60+'])
churn_bar(axes[1,2], k03, 'AgeGroup', 'By Age Group')

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig2_k03_categorical_churn.png')
plt.close()
print('Saved: fig2_k03_categorical_churn.png')

#  FIG 3 — K03: NUMERIC FEATURE DISTRIBUTIONS
num_feats_k03 = ['CreditScore','Age','Tenure','Balance','EstimatedSalary']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Fig 3 — K03: Numeric Feature Distributions by Churn Status', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, feat in enumerate(num_feats_k03):
    for churn_val, color, label in [(0, C_NO, 'No Churn'), (1, C_YES, 'Churn')]:
        data = k03[k03['churn']==churn_val][feat]
        axes[i].hist(data, bins=30, alpha=0.6, color=color, label=label, density=True, edgecolor='white')
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=9)

# Correlation subplot
ax_corr = axes[5]
num_k03_df = k03[['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary','churn']]
corr = num_k03_df.corr()['churn'].drop('churn').sort_values()
colors_corr = [C_YES if v > 0 else C_NO for v in corr.values]
ax_corr.barh(corr.index, corr.values, color=colors_corr, edgecolor='white')
ax_corr.axvline(0, color='gray', linewidth=0.8)
ax_corr.set_title('Correlation with Churn', fontsize=11, fontweight='bold')
ax_corr.set_xlabel('Pearson r')
for i, (val, name) in enumerate(zip(corr.values, corr.index)):
    ax_corr.text(val + (0.003 if val >= 0 else -0.003), i,
                 f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig3_k03_numeric_distributions.png')
plt.close()
print('Saved: fig3_k03_numeric_distributions.png')

#  FIG 4 — K04: CHURN BY CATEGORICAL FEATURES
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Fig 4 — K04: Churn Rate by Categorical Features', fontsize=14, fontweight='bold')

churn_bar(axes[0,0], k04, 'Contract',        'By Contract Type',     rotate=True)
churn_bar(axes[0,1], k04, 'InternetService', 'By Internet Service')
churn_bar(axes[0,2], k04, 'PaymentMethod',   'By Payment Method',    rotate=True)
churn_bar(axes[1,0], k04, 'SeniorCitizen',   'By Senior Citizen (0=No, 1=Yes)')
churn_bar(axes[1,1], k04, 'Partner',         'By Partner Status')
churn_bar(axes[1,2], k04, 'Dependents',      'By Dependents Status')

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig4_k04_categorical_churn.png')
plt.close()
print('Saved: fig4_k04_categorical_churn.png')

#  FIG 5 — K04: NUMERIC DISTRIBUTIONS + TENURE GROUP
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Fig 5 — K04: Numeric Feature Distributions by Churn Status', fontsize=14, fontweight='bold')
axes = axes.flatten()

num_feats_k04 = ['tenure','MonthlyCharges','TotalCharges']
for i, feat in enumerate(num_feats_k04):
    for churn_val, color, label in [(0, C_NO, 'No Churn'), (1, C_YES, 'Churn')]:
        data = k04[k04['churn']==churn_val][feat]
        axes[i].hist(data, bins=30, alpha=0.6, color=color, label=label, density=True, edgecolor='white')
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=9)

# Tenure group bar
k04['TenureGroup'] = pd.cut(k04['tenure'], bins=[-1,12,24,48,72], labels=['0-12m','13-24m','25-48m','49-72m'])
tg = k04.groupby('TenureGroup', observed=True)['churn'].mean()*100
colors_tg = [C_YES if v > 35 else C_AMB if v > 20 else C_TEAL for v in tg.values]
bars = axes[3].bar(tg.index.astype(str), tg.values, color=colors_tg, edgecolor='white')
for bar, val in zip(bars, tg.values):
    axes[3].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[3].set_title('Churn Rate by Tenure Group', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Churn Rate (%)')
axes[3].set_ylim(0, 60)
axes[3].axhline(k04['churn'].mean()*100, color='gray', linestyle='--', linewidth=1, alpha=0.7)

# Correlation
ax_corr4 = axes[4]
num_k04_df = k04[['SeniorCitizen','tenure','MonthlyCharges','TotalCharges','churn']]
corr4 = num_k04_df.corr()['churn'].drop('churn').sort_values()
colors_corr4 = [C_YES if v > 0 else C_NO for v in corr4.values]
ax_corr4.barh(corr4.index, corr4.values, color=colors_corr4, edgecolor='white')
ax_corr4.axvline(0, color='gray', linewidth=0.8)
ax_corr4.set_title('Correlation with Churn', fontsize=11, fontweight='bold')
ax_corr4.set_xlabel('Pearson r')
for i, (val, name) in enumerate(zip(corr4.values, corr4.index)):
    ax_corr4.text(val+(0.005 if val>=0 else -0.005), i,
                  f'{val:.3f}', va='center', ha='left' if val>=0 else 'right', fontsize=9)

# MonthlyCharges boxplot by churn
ax_box = axes[5]
data_no  = k04[k04['churn']==0]['MonthlyCharges']
data_yes = k04[k04['churn']==1]['MonthlyCharges']
bp = ax_box.boxplot([data_no, data_yes], patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], CMAP):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax_box.set_xticklabels(['No Churn','Churn'])
ax_box.set_title('MonthlyCharges Distribution', fontsize=11, fontweight='bold')
ax_box.set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig5_k04_numeric_distributions.png')
plt.close()
print('Saved: fig5_k04_numeric_distributions.png')

#  FIG 6 — MISSING VALUES & DATA QUALITY SUMMARY
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Fig 6 — Data Quality Summary', fontsize=14, fontweight='bold')

for ax, df, name in zip(axes, [k03, k04], ['K03', 'K04']):
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if len(miss) == 0:
        ax.text(0.5, 0.5, 'No missing values\n✓ Complete dataset',
                ha='center', va='center', fontsize=14, color=C_TEAL, fontweight='bold',
                transform=ax.transAxes)
        ax.set_title(f'{name}: Missing Values', fontsize=12, fontweight='bold')
        ax.axis('off')
    else:
        ax.barh(miss.index, miss.values, color=C_YES, edgecolor='white')
        for i, val in enumerate(miss.values):
            ax.text(val+0.1, i, str(val), va='center', fontsize=10)
        ax.set_title(f'{name}: Missing Values per Column', fontsize=12, fontweight='bold')
        ax.set_xlabel('Count')

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig6_data_quality.png')
plt.close()
print('Saved: fig6_data_quality.png')

#  FIG 7 — CORRELATION HEATMAP
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Fig 7 — Feature Correlation Heatmap', fontsize=14, fontweight='bold')

for ax, df, name in zip(axes,
    [k03[['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary','churn']],
     k04[['SeniorCitizen','tenure','MonthlyCharges','TotalCharges','churn']]],
    ['K03','K04']):
    corr_matrix = df.corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, ax=ax, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                linewidths=0.5, annot_kws={'size': 9},
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{name} — Correlation Matrix', fontsize=12, fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig(f'{EDA_DIR}/fig7_correlation_heatmap.png')
plt.close()
print('Saved: fig7_correlation_heatmap.png')

# ─── SUMMARY ──────────────────────────────────────────────────────────────
print()
print('═'*55)
print('  EDA SELESAI ✅')
print('═'*55)
print(f'  Semua gambar tersimpan di: {EDA_DIR}')
print()
print('  K03 Summary:')
print(f'    Samples     : {len(k03):,}')
print(f'    Features    : {k03.shape[1]}')
print(f'    Churn rate  : {k03["churn"].mean()*100:.1f}%  ({k03["churn"].sum():,} churned)')
print(f'    IR          : {(k03["churn"]==0).sum()/(k03["churn"]==1).sum():.2f}')
print(f'    Missing     : {k03.isnull().sum().sum()}')
print()
print('  K04 Summary:')
print(f'    Samples     : {len(k04):,}')
print(f'    Features    : {k04.shape[1]}')
print(f'    Churn rate  : {k04["churn"].mean()*100:.1f}%  ({k04["churn"].sum():,} churned)')
print(f'    IR          : {(k04["churn"]==0).sum()/(k04["churn"]==1).sum():.2f}')
print(f'    Missing     : {k04["TotalCharges"].isna().sum()} (TotalCharges, imputed with median)')
print('═'*55)


Saved: fig1_class_distribution.png


/tmp/ipykernel_58/3185675135.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tbl = df.groupby(col)['churn'].mean().sort_values(ascending=False)*100


Saved: fig2_k03_categorical_churn.png
Saved: fig3_k03_numeric_distributions.png
Saved: fig4_k04_categorical_churn.png
Saved: fig5_k04_numeric_distributions.png
Saved: fig6_data_quality.png
Saved: fig7_correlation_heatmap.png

═══════════════════════════════════════════════════════
  EDA SELESAI ✅
═══════════════════════════════════════════════════════
  Semua gambar tersimpan di: /kaggle/working/hbp_results2/eda

  K03 Summary:
    Samples     : 10,000
    Features    : 16
    Churn rate  : 20.4%  (2,037 churned)
    IR          : 3.91
    Missing     : 0

  K04 Summary:
    Samples     : 7,043
    Features    : 23
    Churn rate  : 26.5%  (1,869 churned)
    IR          : 2.77
    Missing     : 0 (TotalCharges, imputed with median)
═══════════════════════════════════════════════════════


# Configuration

## Seed Locking 

In [ ]:
import os, random

SEED = 42
os.environ['PYTHONHASHSEED']        = str(SEED)
os.environ['TF_DETERMINISTIC_OPS']  = '1'
os.environ['TF_CUDNN_DETERMINISTIC']= '1'
os.environ['TF_CPP_MIN_LOG_LEVEL']  = '2'

random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(4)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU tersedia: {len(tf.config.list_physical_devices("GPU"))} unit')
print('Seed terkunci: 42')

2026-06-16 07:04:52.995757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781593493.553476      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781593493.712528      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781593495.266674      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781593495.266750      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781593495.266755      58 computation_placer.cc:177] computation placer alr

TensorFlow : 2.19.0
GPU tersedia: 0 unit
Seed terkunci: 42


2026-06-16 07:05:22.066203: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Import Library

In [18]:
import sys, json, logging, warnings, gc
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.metrics import precision_score, recall_score
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
import optuna
from pyswarms.single import GlobalBestPSO
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(f'{RESULTS_DIR}/pipeline.log'),
        logging.StreamHandler(sys.stdout),
    ]
)
log = logging.getLogger(__name__)
print('Import selesai')

Import selesai


## Konfigurasi Testing

In [ ]:
# ─── HYPERPARAMETER SEARCH SPACE ─────────────────────────────
HP_SPACE = {
    'n_layers'  : {'type': 'int',         'low': 1,    'high': 5},
    'n_neurons' : {'type': 'int_log',     'low': 16,   'high': 512},
    'dropout'   : {'type': 'float',       'low': 0.0,  'high': 0.4},
    'lr'        : {'type': 'float_log',   'low': 1e-5, 'high': 1e-1},
    'batch_size': {'type': 'categorical', 'choices': [16, 32, 64, 128]},
    'epochs'    : {'type': 'int',         'low': 50,   'high': 300},
}

HPO_EPOCHS   = 30    # untuk PSO dan BO phase
FINAL_EPOCHS = 100   # untuk evaluasi akhir (C1-C4)


# ─── EVALUASI AKHIR ────────────────────────────────────────────────────────
EVAL_CV_FOLDS = 10 
GENERAL_CV_FOLDS = 5

# ─── PSO ───────────────────────────────────────────────
PSO_N_PARTICLES = 10
PSO_ITERATIONS  = 15   # C2: 10 × 15 = 150 eval
PSO_OPTIONS     = {'c1': 1.5, 'c2': 1.5, 'w': 0.72}
PSO_CV_FOLDS    = GENERAL_CV_FOLDS
PSO_TOP_K       = 5
PSO_MIN_BOUND = np.array([1, np.log(16),  0.0, np.log10(1e-5), 0.0,  20.0])
PSO_MAX_BOUND = np.array([5, np.log(512), 0.4, np.log10(1e-1), 3.999, 20.0])
BATCH_SIZES     = [64,128,256,512]

# ─── BO ────────────────────────────────────────────────
BO_N_WARMSTART = PSO_TOP_K  
BO_N_TRIALS    = 25
BO_CV_FOLDS    = GENERAL_CV_FOLDS

#   C2 (PSO) : 150 eval PSO
#   C3 (BO)  : 150 trial TPE  
#   C4 (HBP) : 90 eval PSO + 60 trial BO = 150   
TOTAL_BUDGET     = PSO_N_PARTICLES * PSO_ITERATIONS   # 150
PSO_C4_PARTICLES = 10
PSO_C4_ITERS     = 9                                  # 10 × 9  = 90 eval
BO_C4_WARMSTART  = 5
BO_C4_TRIALS     = 55                                 # 5 + 55 = 60 trial
BO_C4_STARTUP    = BO_C4_WARMSTART                    # TPE mulai setelah warm-start
assert PSO_C4_PARTICLES * PSO_C4_ITERS + BO_C4_WARMSTART + BO_C4_TRIALS == TOTAL_BUDGET, \
       'Budget C4 tidak sama dengan C2/C3 — perbaiki konstanta!'

USE_REFIT_SELECTION = True   
REFIT_TOPK          = 3      
REFIT_CV_FOLDS      = EVAL_CV_FOLDS   

WARMSTART_FORCE_BEST = 3     
WARMSTART_MIN_DIST   = 0.25

# ─── BASELINE C1 (Gkonis 2025) ─────────────────────────────────────────────
C1_CONFIG = {
    'n_layers': 2, 'n_neurons': 7, 'dropout': 0.0,
    'lr': 1e-3, 'batch_size': 32, 'epochs': 100,
}

print('═' * 60)
print('  PRODUCTION CONFIG LOADED ')
print('═' * 60)
print(f'  Datasets       : {list(DATASETS.keys())}')
print('-' * 60)
print(f'  HP Space       : layers 1-5, neurons 16-512, lr 1e-5–1e-1')
print('-' * 60)
print(f'  Total budget / kondisi : {TOTAL_BUDGET} evaluasi (disamakan)')
print(f'  C2 PSO   : {PSO_N_PARTICLES}×{PSO_ITERATIONS} = {PSO_N_PARTICLES*PSO_ITERATIONS} eval')
print(f'  C3 BO    : {PSO_N_PARTICLES*PSO_ITERATIONS} trial TPE (startup 5)')
print(f'  C4 HBP   : {PSO_C4_PARTICLES}×{PSO_C4_ITERS}={PSO_C4_PARTICLES*PSO_C4_ITERS} PSO '
      f'+ ({BO_C4_WARMSTART}+{BO_C4_TRIALS})={BO_C4_WARMSTART+BO_C4_TRIALS} BO '
      f'= {PSO_C4_PARTICLES*PSO_C4_ITERS + BO_C4_WARMSTART + BO_C4_TRIALS}')
print('-' * 60)
print(f'  Proxy HPO : {HPO_EPOCHS} epoch / {PSO_CV_FOLDS}-fold  |  Final: {FINAL_EPOCHS} epoch / {EVAL_CV_FOLDS}-fold')
print(f'  Refit-select: {USE_REFIT_SELECTION} (top-{REFIT_TOPK} @ {REFIT_CV_FOLDS}-fold)')
print(f'  Final eval     : {EVAL_CV_FOLDS}-fold stratified CV')
print('═' * 60)


════════════════════════════════════════════════════════════
  PRODUCTION CONFIG LOADED 
════════════════════════════════════════════════════════════
  Datasets       : ['K03', 'K04']
------------------------------------------------------------
  HP Space       : layers 1-5, neurons 16-512, lr 1e-5–1e-1
------------------------------------------------------------
  Total budget / kondisi : 150 evaluasi (disamakan)
  C2 PSO   : 10×15 = 150 eval
  C3 BO    : 150 trial TPE (startup 5)
  C4 HBP   : 10×9=90 PSO + (5+55)=60 BO = 150
------------------------------------------------------------
  Proxy HPO : 30 epoch / 5-fold  |  Final: 100 epoch / 10-fold
  Refit-select: True (top-3 @ 10-fold)
  Final eval     : 10-fold stratified CV
════════════════════════════════════════════════════════════


## APTx Custom Activation

In [20]:
class APTxActivation(tf.keras.layers.Layer):
    """APTx: f(x) = alpha * x * tanh(beta * softplus(x)) — Kumar 2022"""
    def __init__(self, **kwargs): super().__init__(**kwargs)

    def build(self, input_shape):
        self.alpha = self.add_weight(
            name='alpha', shape=(), initializer='ones', trainable=True
        )
        self.beta = self.add_weight(
            name='beta', shape=(), initializer='ones', trainable=True
        )
        self.built = True

    def call(self, x):
        return self.alpha * x * tf.math.tanh(self.beta * tf.math.softplus(x))

    def get_config(self): return super().get_config()

## DNN Builder

In [21]:
def build_dnn(input_dim: int, config: dict) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,))
    x = inputs
    for i in range(config['n_layers']):
        x = tf.keras.layers.Dense(
            config['n_neurons'],
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED)
        )(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = APTxActivation()(x)
        if config['dropout'] > 0:
            x = tf.keras.layers.Dropout(config['dropout'], seed=SEED)(x)
    output = tf.keras.layers.Dense(
        1, activation='sigmoid',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED)
    )(x)
    model = tf.keras.Model(inputs, output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config['lr']),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    return model

# Quick test
m = build_dnn(12, C1_CONFIG)
print(f'DNN builder OK — {m.count_params()} params untuk C1 config')

DNN builder OK — 215 params untuk C1 config


## Load & Preprocess Data

In [ ]:
def load_and_preprocess(dataset_key: str):
    cfg = DATASETS[dataset_key]
    log.info(f'[{dataset_key}] Memuat: {cfg["name"]}')
    df = pd.read_csv(cfg['path'])
    log.info(f'[{dataset_key}] Ukuran awal: {df.shape}')

    if 'TotalCharges' in df.columns:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        n_na = df['TotalCharges'].isna().sum()
        if n_na > 0:
            median_tc = df['TotalCharges'].median()
            df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
            log.info(f'[{dataset_key}] TotalCharges: {n_na} nilai kosong diisi median={median_tc:.2f}')

    n_before = len(df)
    df = df.drop_duplicates()
    if len(df) < n_before:
        log.info(f'[{dataset_key}] Hapus {n_before-len(df)} duplikat')

    target_col = cfg['target']
    if cfg.get('positive_val'):
        df[target_col] = (df[target_col] == cfg['positive_val']).astype(int)
    y = df[target_col].values.astype(int)

    cols_to_drop = cfg.get('drop_cols', []) + [target_col]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    for col in cat_cols:
        df = pd.get_dummies(df, columns=[col], drop_first=False)

    feature_names = df.columns.tolist()
    X = df.values.astype(np.float32)
    ir = (y==0).sum() / max((y==1).sum(), 1)
    log.info(f'[{dataset_key}] Ukuran akhir: {X.shape}, IR={ir:.2f}')
    return X, y, cfg['name'], feature_names

print('load_and_preprocess OK ✅')


load_and_preprocess OK ✅


## Fungsi Evaluasi Konfigurasi

In [ ]:
import time
from sklearn.model_selection import train_test_split
def evaluate_config(config, X, y, cv_folds=3, verbose=False, max_rows=None):
    if max_rows and len(X) > max_rows:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(X), size=max_rows, replace=False)
        X, y = X[idx], y[idx]

    skf  = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=SEED)
    aucs = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        X_tr2, X_es, y_tr2, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED
        )

        scaler = RobustScaler()
        X_tr2  = scaler.fit_transform(X_tr2)
        X_es   = scaler.transform(X_es)
        X_val  = scaler.transform(X_val)

        try:
            smt = SMOTETomek(smote=SMOTE(k_neighbors=5, random_state=SEED), random_state=SEED)
            X_tr2, y_tr2 = smt.fit_resample(X_tr2, y_tr2)
        except Exception as e:
            if verbose: print(f'  SMOTETomek fallback fold {fold_idx}: {e}')

        model = build_dnn(X_tr2.shape[1], config)
        _t = time.perf_counter()
        history = model.fit(
            X_tr2, y_tr2,
            validation_data=(X_es, y_es),
            batch_size=config['batch_size'],
            epochs=config['epochs'],
            callbacks=[tf.keras.callbacks.EarlyStopping(
                monitor='val_auc', patience=5, min_delta=0.001,
                restore_best_weights=True, mode='max'
            )],
            verbose=0,
        )
        _dt   = time.perf_counter() - _t
        n_ep  = len(history.history['loss'])
        y_prob = model.predict(X_val, verbose=0).ravel()
        aucs.append(roc_auc_score(y_val, y_prob))

        if verbose:
            es = 'ES AKTIF' if n_ep < config['epochs'] else 'PENUH (ES tdk aktif!)'
            print(f'  [fold {fold_idx}] {n_ep:>2}/{config["epochs"]} epoch ({es}) | '
                  f'{_dt:5.1f}s | AUC={aucs[-1]:.4f}')

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    return float(np.mean(aucs))

print('evaluate_config OK')


evaluate_config (v3.1: + logging per-fold) OK


## Helper Konvergensi

In [24]:
import matplotlib.pyplot as plt

def running_best(values, seed=-np.inf):
    best, out = seed, []
    for v in values:
        best = max(best, v)
        out.append(best)
    return out

def plot_convergence_curves(traces: dict, title='Konvergensi: Best Proxy-AUC vs Jumlah Evaluasi'):
    plt.figure(figsize=(8, 5))
    for label, trace in traces.items():
        plt.plot(range(1, len(trace) + 1), trace, label=label, linewidth=2)
    plt.xlabel('Evaluasi ke-')
    plt.ylabel('Best proxy-AUC sejauh ini')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

print('Helper konvergensi (running_best, plot_convergence_curves) OK')

Helper konvergensi (running_best, plot_convergence_curves) OK


## PSO Phase 

In [ ]:
def decode_pso_position(pos):
    return {
        'n_layers' : int(np.clip(np.round(pos[0]), 1, 3)),
        'n_neurons': int(np.clip(np.round(np.exp(pos[1])), 16, 128)),
        'dropout'   : float(np.clip(pos[2], 0.0, 0.4)),
        'lr'       : float(10 ** np.clip(pos[3], -4.0, -2.0)),
        'batch_size': BATCH_SIZES[int(np.clip(int(pos[4]), 0, len(BATCH_SIZES)-1))],
        'epochs': HPO_EPOCHS,
    }

def run_pso_phase(X, y, n_particles=PSO_N_PARTICLES, n_iters=PSO_ITERATIONS, max_rows=None):
    log.info(f'  [PSO] {n_particles} partikel x {n_iters} iterasi')
    all_positions, all_scores = [], []

    def pso_objective(positions):
        costs = np.zeros(len(positions))
        for i, pos in enumerate(positions):
            config = decode_pso_position(pos)
            auc    = evaluate_config(config, X, y, cv_folds=PSO_CV_FOLDS, max_rows=max_rows)
            costs[i] = -auc
            all_positions.append(pos.copy())
            all_scores.append(auc)
        return costs

    optimizer = GlobalBestPSO(
        n_particles=n_particles,
        dimensions=len(PSO_MIN_BOUND),
        options=PSO_OPTIONS,
        bounds=(PSO_MIN_BOUND, PSO_MAX_BOUND),
    )
    optimizer.optimize(pso_objective, iters=n_iters, verbose=True)

    scored = sorted(zip(all_scores, all_positions), key=lambda x: -x[0])
    seen, top5_configs, top5_scores = set(), [], []
    for score, pos in scored:
        cfg = decode_pso_position(pos)
        key = tuple(cfg.values())
        if key not in seen:
            seen.add(key); top5_configs.append(cfg); top5_scores.append(score)
        if len(top5_configs) == PSO_TOP_K: break
    while len(top5_configs) < PSO_TOP_K:
        top5_configs.append(top5_configs[-1]); top5_scores.append(top5_scores[-1])

    trace = running_best(all_scores)

    log.info(f'  [PSO] Top-5 AUC: {[f"{s:.4f}" for s in top5_scores]} | '
             f'final best-so-far: {trace[-1]:.4f}')
    return top5_configs, top5_scores, trace

print('run_pso_phase (+ trace) OK')


run_pso_phase (+ trace) OK


## Bayesian Optimization Phase 

In [26]:
def run_bo_phase(X, y, warmstart_configs,
                 n_warmstart=None, n_trials=None, n_startup=None,
                 max_rows=None, return_topk=None):
    n_warmstart    = BO_N_WARMSTART if n_warmstart is None else n_warmstart
    n_trials_total = (BO_N_WARMSTART + BO_N_TRIALS) if n_trials is None else n_trials
    n_startup      = n_warmstart if n_startup is None else n_startup
    return_topk    = (REFIT_TOPK if 'REFIT_TOPK' in globals() else 3) \
                     if return_topk is None else return_topk

    log.info(f'  [BO] {n_warmstart} warm-start + {n_trials_total - n_warmstart} guided '
             f'(startup={n_startup}) = {n_trials_total} trial TPE')

    def objective(trial):
        config = {
            'n_layers'  : trial.suggest_int('n_layers', 1, 3),
            'n_neurons' : trial.suggest_int('n_neurons', 16, 128, log=True),
            'dropout'   : trial.suggest_float('dropout', 0.0, 0.4),
            'lr'        : trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', BATCH_SIZES),
            'epochs'    : HPO_EPOCHS,
        }
        return evaluate_config(config, X, y, cv_folds=BO_CV_FOLDS, max_rows=max_rows)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(n_startup_trials=n_startup, seed=SEED))

    for cfg in warmstart_configs[:n_warmstart]:
        study.enqueue_trial({
            'n_layers'  : int(cfg['n_layers']),
            'n_neurons' : int(cfg['n_neurons']),
            'dropout'   : float(cfg['dropout']),
            'lr'        : float(cfg['lr']),
            'batch_size': int(cfg['batch_size']),
        })

    study.optimize(objective, n_trials=n_trials_total, show_progress_bar=True)

    done = [t for t in study.trials if t.value is not None]
    done.sort(key=lambda t: t.value, reverse=True)
    topk = [{**t.params, 'epochs': HPO_EPOCHS} for t in done[:return_topk]]

    # ★ BARU — jejak konvergensi kronologis (urut nomor trial, BUKAN urut skor)
    chronological = sorted([t for t in study.trials if t.value is not None],
                            key=lambda t: t.number)
    chrono_values = [t.value for t in chronological]

    log.info(f'  [BO] Best proxy-AUC: {study.best_value:.4f} — {study.best_params}')
    return study.best_params, study.best_value, topk, chrono_values

print('run_bo_phase (fair + top-k + trace) OK')


run_bo_phase (fair + top-k + trace) OK


# Empat Kondisi Eksperimen

## C1 - C3

In [ ]:
def run_condition_c1(X, y, **_):
    log.info('  [C1] Baseline manual — konfigurasi Gkonis 2025')
    config = {**C1_CONFIG, 'epochs': FINAL_EPOCHS}
    auc = evaluate_config(config, X, y, cv_folds=EVAL_CV_FOLDS)
    return config, auc, None       


def run_condition_c2(X, y, ds_key=None, **_):
    log.info('  [C2] PSO tunggal')
    configs, scores, trace = run_pso_phase(X, y)         
    best = int(np.argmax(scores))
    config = {**configs[best], 'epochs': FINAL_EPOCHS}
    if ds_key:
        save_checkpoint(ds_key, 'C2_pso', config,
                        {'auc': [float(scores[best])], 'convergence_trace': trace})
    auc = evaluate_config(config, X, y, cv_folds=EVAL_CV_FOLDS)
    return config, auc, trace


def run_condition_c3(X, y, ds_key=None, **_):
    log.info(f'  [C3] Bayesian Optimization tunggal — {PSO_N_PARTICLES * PSO_ITERATIONS} trial')

    best_so_far = {'value': -np.inf, 'params': None}

    def objective(trial):
        config = {
            'n_layers'  : trial.suggest_int('n_layers', 1, 3),
            'n_neurons' : trial.suggest_int('n_neurons', 16, 128, log=True),
            'dropout'   : trial.suggest_float('dropout', 0.0, 0.4),
            'lr'        : trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', BATCH_SIZES),
            'epochs'    : HPO_EPOCHS,
        }
        auc = evaluate_config(config, X, y, cv_folds=BO_CV_FOLDS)
        if auc > best_so_far['value']:
            best_so_far['value'] = auc
            best_so_far['params'] = config
            if ds_key:
                save_checkpoint(ds_key, 'C3_best_so_far', config, {'auc': [float(auc)]})
            log.info(f'  [C3] Trial {trial.number} baru terbaik: {auc:.4f}')
        return auc

    n_c3 = PSO_N_PARTICLES * PSO_ITERATIONS
    study = optuna.create_study(direction='maximize',
                sampler=optuna.samplers.TPESampler(n_startup_trials=5, seed=SEED))
    study.optimize(objective, n_trials=n_c3, show_progress_bar=True)

    chronological = sorted([t for t in study.trials if t.value is not None],
                            key=lambda t: t.number)
    trace = running_best([t.value for t in chronological])

    config = {**study.best_params, 'epochs': FINAL_EPOCHS}
    auc = evaluate_config(config, X, y, cv_folds=EVAL_CV_FOLDS)
    return config, auc, trace

print('run_condition_c1/c2/c3 (+ trace) OK')

run_condition_c1/c2/c3 (+ trace) OK


## C4

In [ ]:
def config_to_vector(cfg):
    return np.array([
        (cfg['n_layers'] - 1) / (3 - 1),
        (np.log(cfg['n_neurons']) - np.log(16)) / (np.log(128) - np.log(16)),
        cfg['dropout'] / 0.4,
        (np.log10(cfg['lr']) - (-4)) / ((-2) - (-4)),
        BATCH_SIZES.index(int(cfg['batch_size'])) / (len(BATCH_SIZES) - 1),
    ])

def select_diverse_topk(configs, scores, k=5,
                        min_dist=WARMSTART_MIN_DIST,
                        n_force_best=WARMSTART_FORCE_BEST):
    order   = list(np.argsort(scores)[::-1])
    vectors = [config_to_vector(c) for c in configs]
    selected = list(order[:min(n_force_best, k)])
    for idx in order[len(selected):]:
        if len(selected) >= k:
            break
        if min(np.linalg.norm(vectors[idx] - vectors[s]) for s in selected) > min_dist:
            selected.append(idx)
    for idx in order:
        if len(selected) >= k:
            break
        if idx not in selected:
            selected.append(idx)
    log.info(f'  [C4] Warm-start terpilih (indeks {selected}), '
             f'AUC: {[f"{scores[i]:.4f}" for i in selected]}')
    return [configs[i] for i in selected]


def _run_pso_c4(X, y, n_particles=PSO_C4_PARTICLES, n_iters=PSO_C4_ITERS, max_rows=None):
    log.info(f'  [C4-PSO] {n_particles} partikel × {n_iters} iterasi = {n_particles*n_iters} eval')
    all_configs, all_scores = [], []

    def pso_objective(positions):
        costs = np.zeros(len(positions))
        for i, pos in enumerate(positions):
            cfg = decode_pso_position(pos)
            auc = evaluate_config(cfg, X, y, cv_folds=PSO_CV_FOLDS, max_rows=max_rows)
            costs[i] = -auc
            all_configs.append(cfg); all_scores.append(auc)
        return costs

    optimizer = GlobalBestPSO(
        n_particles=n_particles, dimensions=len(PSO_MIN_BOUND),
        options=PSO_OPTIONS, bounds=(PSO_MIN_BOUND, PSO_MAX_BOUND))
    optimizer.optimize(pso_objective, iters=n_iters, verbose=True)

    trace = running_best(all_scores)               

    best_of = {}
    for cfg, s in zip(all_configs, all_scores):
        key = tuple(cfg.values())
        if key not in best_of or s > best_of[key][1]:
            best_of[key] = (cfg, s)
    uniq    = list(best_of.values())
    configs = [c for c, _ in uniq]
    scores  = [s for _, s in uniq]
    log.info(f'  [C4-PSO] {len(configs)} config unik — best AUC: {max(scores):.4f}')
    return configs, scores, trace


def final_select_config(candidates, X, y, cv_folds=REFIT_CV_FOLDS):
    best_cfg, best_score, seen = None, -np.inf, set()
    for c in candidates:
        cfg = {**c, 'epochs': FINAL_EPOCHS}
        key = (cfg['n_layers'], cfg['n_neurons'], round(float(cfg['dropout']), 4),
               round(float(cfg['lr']), 6), int(cfg['batch_size']))
        if key in seen:
            continue
        seen.add(key)
        s = evaluate_config(cfg, X, y, cv_folds=cv_folds)
        log.info(f'  [C4-refit] AUC={s:.4f} ← {key}')
        if s > best_score:
            best_score, best_cfg = s, cfg
    return best_cfg, best_score


def run_condition_c4(X, y, ds_key=None, **_):
    log.info('  [C4] HBP fair-budget: PSO(90) → diverse warm-start → BO-TPE(60) → refit-select')

    if ds_key:
        saved_cfg, saved_m = load_checkpoint(ds_key, 'C4_bo')
        if saved_cfg:
            log.info('  [C4] Resume dari checkpoint C4_bo — skip PSO & BO')
            config = {**saved_cfg, 'epochs': FINAL_EPOCHS}
            auc = evaluate_config(config, X, y, cv_folds=EVAL_CV_FOLDS)
            trace = saved_m.get('convergence_trace', []) 
            return config, auc, trace

    # ── FASE 1: PSO (budget C4) ───────────────────────────────────────────
    pso_configs, pso_scores, pso_trace = None, None, None
    if ds_key:
        c, m = load_checkpoint(ds_key, 'C4_pso')
        if c and 'pool_configs' in c:
            log.info('  [C4] Resume fase PSO dari C4_pso')
            pso_configs = c['pool_configs']
            pso_scores  = m['auc']
            pso_trace   = m.get('convergence_trace', running_best(pso_scores))
    if pso_configs is None:
        pso_configs, pso_scores, pso_trace = _run_pso_c4(X, y)
        if ds_key:
            save_checkpoint(ds_key, 'C4_pso', {'pool_configs': pso_configs},
                            {'auc': [float(s) for s in pso_scores],
                             'convergence_trace': pso_trace})

    diverse = select_diverse_topk(pso_configs, pso_scores,
                                  k=BO_C4_WARMSTART,
                                  min_dist=WARMSTART_MIN_DIST,
                                  n_force_best=WARMSTART_FORCE_BEST)

    # ── FASE 2: BO (60 trial, startup=5) ───────────────────────────────────
    bo_best_params, bo_best_val, bo_topk, bo_chrono = run_bo_phase(
        X, y, diverse,
        n_warmstart=BO_C4_WARMSTART,
        n_trials=BO_C4_WARMSTART + BO_C4_TRIALS,
        n_startup=BO_C4_STARTUP,
        return_topk=REFIT_TOPK,
    )

    bo_trace_continued = running_best(bo_chrono, seed=pso_trace[-1])
    full_trace = pso_trace + bo_trace_continued   # panjang = 90 + 60 = 150

    best_pso = max(zip(pso_scores, pso_configs), key=lambda t: t[0])[1]
    if USE_REFIT_SELECTION:
        candidates  = bo_topk + [best_pso]
        config, auc = final_select_config(candidates, X, y)
    else:
        config = {**bo_best_params, 'epochs': FINAL_EPOCHS}
        auc    = evaluate_config(config, X, y, cv_folds=EVAL_CV_FOLDS)

    if ds_key:
        save_checkpoint(ds_key, 'C4_bo', config,
                        {'auc': [float(auc)], 'convergence_trace': full_trace})
    log.info(f'  [C4] Config final: {config} — AUC(10-fold)={auc:.4f} | '
             f'best proxy-AUC sepanjang pencarian: {full_trace[-1]:.4f}')
    return config, auc, full_trace

print('run_condition_c4 (fair-budget + diverse warm-start + refit-select + trace) OK')

run_condition_c4 (fair-budget + diverse warm-start + refit-select + trace) OK


# Evaluasi 

## Final Evaluation

In [29]:
def final_evaluation(config, X, y, condition, dataset_key):
    log.info(f'  [{condition}] Evaluasi akhir {EVAL_CV_FOLDS}-fold CV...')
    skf     = StratifiedKFold(n_splits=EVAL_CV_FOLDS, shuffle=True, random_state=SEED)
    metrics = {'auc': [], 'f1': [], 'accuracy': [], 'precision': [], 'recall': []}
    _epochs_log = []

    for fold_idx, (train_idx, val_idx) in enumerate(tqdm(
        skf.split(X, y), total=EVAL_CV_FOLDS, desc=f'[{condition}] 10-fold'
    )):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        X_tr2, X_es, y_tr2, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED
        )

        scaler = RobustScaler()
        X_tr2  = scaler.fit_transform(X_tr2)
        X_es   = scaler.transform(X_es)
        X_val  = scaler.transform(X_val)

        try:
            smt = SMOTETomek(smote=SMOTE(k_neighbors=5, random_state=SEED), random_state=SEED)
            X_tr2, y_tr2 = smt.fit_resample(X_tr2, y_tr2)
        except Exception:
            pass

        model = build_dnn(X_tr2.shape[1], config)
        history = model.fit(
            X_tr2, y_tr2,
            validation_data=(X_es, y_es),
            batch_size=config['batch_size'],
            epochs=config['epochs'],
            callbacks=[tf.keras.callbacks.EarlyStopping(
                monitor='val_auc', patience=5, min_delta=0.001,
                restore_best_weights=True, mode='max')],
            verbose=0,
        )
        _epochs_log.append(len(history.history['loss']))

        y_prob = model.predict(X_val, verbose=0).ravel()
        y_pred = (y_prob >= 0.5).astype(int)

        metrics['auc'].append(roc_auc_score(y_val, y_prob))
        metrics['f1'].append(f1_score(y_val, y_pred, average='macro', zero_division=0))
        metrics['accuracy'].append(accuracy_score(y_val, y_pred))
        metrics['precision'].append(precision_score(y_val, y_pred, average='macro', zero_division=0))
        metrics['recall'].append(recall_score(y_val, y_pred, average='macro', zero_division=0))

        del model; tf.keras.backend.clear_session(); gc.collect()

    n_full = sum(1 for e in _epochs_log if e >= config['epochs'])
    log.info(f'  [{condition}] AUC: {np.mean(metrics["auc"]):.4f} +- {np.std(metrics["auc"]):.4f} | '
             f'epoch/fold={_epochs_log} | {n_full}/{EVAL_CV_FOLDS} fold jalan PENUH')
    return metrics

print('final_evaluation (v3.1: + logging per-fold) OK')


final_evaluation (v3.1: + logging per-fold) OK


## Uji Statistik (Friedman + Wilcoxon + Holm-Bonferroni)

In [ ]:
def run_statistical_tests(auc_results):
    conditions = ['C1','C2','C3','C4']
    aucs = [auc_results[c] for c in conditions]
    stat_f, p_f = friedmanchisquare(*aucs)
    log.info(f'  [STAT] Friedman: chi2={stat_f:.4f}, p={p_f:.6f} ({"SIGNIFIKAN" if p_f<0.05 else "tidak sig"})')

    pairs = [('C1','C2'),('C1','C3'),('C1','C4'),('C2','C3'),('C2','C4'),('C3','C4')]
    raw_p, stats_w = [], []
    for ca, cb in pairs:
        s, p = wilcoxon(auc_results[ca], auc_results[cb], alternative='two-sided')
        raw_p.append(p); stats_w.append(s)

    reject, p_corr, _, _ = multipletests(raw_p, method='holm', alpha=0.05)
    n = len(aucs[0])

    # Friedman ranking — peringkat rata-rata per kondisi
    matrix = np.array(aucs).T  # (n_folds, n_conditions)
    from scipy.stats import rankdata as _rankdata
    ranks = np.apply_along_axis(lambda row: _rankdata(-row), axis=1, arr=matrix)
    mean_ranks = {c: round(float(np.mean(ranks[:,j])),3) for j,c in enumerate(conditions)}

    result = {
        'friedman': {'statistic': round(stat_f,4), 'p_value': round(p_f,6),
                     'significant': bool(p_f<0.05)},
        'mean_friedman_rank': mean_ranks,
        'pairwise': []
    }

    for i,(ca,cb) in enumerate(pairs):
        mu    = n * (n + 1) / 4
        sigma = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        Z     = (stats_w[i] - mu) / sigma
        r     = min(abs(Z) / np.sqrt(n), 1.0)  # clip ke max 1.0
        label = 'large' if r >= 0.5 else 'medium' if r >= 0.3 else 'small'

        entry = {
            'pair'        : f'{ca} vs {cb}',
            'p_raw'       : round(raw_p[i], 6),
            'p_holm'      : round(float(p_corr[i]), 6),
            'reject_H0'   : bool(reject[i]),
            'effect_r'    : round(r, 4),
            'effect_label': label,
        }
        result['pairwise'].append(entry)
        log.info(f'  [STAT] {ca} vs {cb}: p_holm={p_corr[i]:.4f}, '
                 f'|r|={r:.3f} ({label}), reject={reject[i]}')

    return result

print('run_statistical_tests OK')


run_statistical_tests OK


# Checkpoint System

In [31]:
def save_checkpoint(dataset_key, condition, config, metrics):
    path = f'{RESULTS_DIR}/{dataset_key}_{condition}_checkpoint.json'
    with open(path, 'w') as f:
        json.dump({'config': config, 'metrics': metrics}, f, indent=2)
    print(f'Checkpoint disimpan di: {path}')

def load_checkpoint(dataset_key, condition):
    path = f'{RESULTS_DIR}/{dataset_key}_{condition}_checkpoint.json'
    if os.path.exists(path):
        with open(path) as f: data = json.load(f)
        print(f'Checkpoint ditemukan — skip HPO: {path}')
        return data['config'], data['metrics']
    return None, None

print('Checkpoint helper OK')

Checkpoint helper OK


# Eksekusi

In [ ]:
# import time

# DS = 'K03'   
# Xp, yp, _, _ = load_and_preprocess(DS)

# cfg = {'n_layers': 3, 'n_neurons': 128, 'dropout': 0.2,
#        'lr': 1e-3, 'batch_size': 64, 'epochs': HPO_EPOCHS}

# print(f'=== PROBE evaluate_config (jalur HPO) — {DS} ===')
# t = time.perf_counter()
# auc = evaluate_config(cfg, Xp, yp, cv_folds=3, verbose=True)   # verbose -> log per-fold
# dt  = time.perf_counter() - t
# per_fold = dt / 3
# print(f'-> 3 fold: {dt:5.1f}s | {per_fold:4.1f}s/fold | AUC={auc:.4f}')

# est_hpo_min = per_fold * 150 * 3 / 60
# print(f'-> Estimasi HPO 1 kondisi (450 fold): ~{est_hpo_min:.0f} menit')
# print(f'-> Estimasi 3 kondisi HPO (C2+C3+C4): ~{est_hpo_min*3:.0f} menit '
#       f'({est_hpo_min*3/60:.1f} jam) untuk {DS}\n')

# print(f'=== PROBE final_evaluation (jalur final, 10-fold, {FINAL_EPOCHS} epoch) — {DS} ===')
# t = time.perf_counter()
# m = final_evaluation(cfg, Xp, yp, 'PROBE', DS)
# dt = time.perf_counter() - t
# print(f'-> final_evaluation 10-fold: {dt:5.1f}s | AUC={np.mean(m["auc"]):.4f}')
# print(f'-> Estimasi final_eval 4 kondisi: ~{dt*4/60:.0f} menit\n')

# print('CEK INTI:')
# print('  • Tiap baris [fold k] harus "ES AKTIF" (bukan PENUH) -> EarlyStopping hidup.')
# print('  • AUC harus ~0.82-0.87 (bukan ~0.53) -> scaling benar.')


In [ ]:
# import time

# DS = 'K04'   # ganti ke 'K03' kalau mau cek dataset terbesar
# Xp, yp, _, _ = load_and_preprocess(DS)

# cfg = {'n_layers': 3, 'n_neurons': 128, 'dropout': 0.2,
#        'lr': 1e-3, 'batch_size': 64, 'epochs': HPO_EPOCHS}

# print(f'=== PROBE evaluate_config (jalur HPO) — {DS} ===')
# t = time.perf_counter()
# auc = evaluate_config(cfg, Xp, yp, cv_folds=3, verbose=True)   # verbose -> log per-fold
# dt  = time.perf_counter() - t
# per_fold = dt / 3
# print(f'-> 3 fold: {dt:5.1f}s | {per_fold:4.1f}s/fold | AUC={auc:.4f}')

# # Estimasi: 1 kondisi = 150 evaluasi × 3 fold = 450 fold-training
# est_hpo_min = per_fold * 150 * 3 / 60
# print(f'-> Estimasi HPO 1 kondisi (450 fold): ~{est_hpo_min:.0f} menit')
# print(f'-> Estimasi 3 kondisi HPO (C2+C3+C4): ~{est_hpo_min*3:.0f} menit '
#       f'({est_hpo_min*3/60:.1f} jam) untuk {DS}\n')

# print(f'=== PROBE final_evaluation (jalur final, 10-fold, {FINAL_EPOCHS} epoch) — {DS} ===')
# t = time.perf_counter()
# m = final_evaluation(cfg, Xp, yp, 'PROBE', DS)
# dt = time.perf_counter() - t
# print(f'-> final_evaluation 10-fold: {dt:5.1f}s | AUC={np.mean(m["auc"]):.4f}')
# print(f'-> Estimasi final_eval 4 kondisi: ~{dt*4/60:.0f} menit\n')

# print('CEK INTI:')
# print('  • Tiap baris [fold k] harus "ES AKTIF" (bukan PENUH) -> EarlyStopping hidup.')
# print('  • AUC harus ~0.82-0.87 (bukan ~0.53) -> scaling benar.')


In [ ]:
DATASETS_TO_RUN   = ['K04', 'K03']           
CONDITIONS_TO_RUN = ['C1', 'C2', 'C3', 'C4']

CONDITION_RUNNERS = {
    'C1': run_condition_c1,
    'C2': run_condition_c2,
    'C3': run_condition_c3,
    'C4': run_condition_c4,
}

all_results = []

for ds_key in DATASETS_TO_RUN:
    log.info(f'\n{"="*60}')
    log.info(f'DATASET: {ds_key} — {DATASETS[ds_key]["name"]}')
    log.info(f'{"="*60}')

    # Cek file dataset ada
    if not os.path.exists(DATASETS[ds_key]['path']):
        log.warning(f'[SKIP] File tidak ditemukan: {DATASETS[ds_key]["path"]}')
        log.warning(f'       Tambahkan dataset via Add Input di Kaggle')
        continue

    print(f'File ditemukan: {DATASETS[ds_key]["path"]}')
    print(f"\n=== HEAD {ds_key} (RAW) ===")
    df_check = pd.read_csv(DATASETS[ds_key]['path'])
    print(f'Before preprocess: {df_check.columns.tolist()}')
    print(f'Total fitur: {len(df_check.columns.tolist())}')

    X, y, name, feature_names = load_and_preprocess(ds_key)
    print(f'After preprocess shape: {X.shape}')
    print(f'Feature names after preprocess: {feature_names}')
    print(f'Total fitur: {len(feature_names)}')

    max_rows   = DATASETS[ds_key].get('hpo_max_rows', None)
    X_hpo = X[:max_rows] if max_rows and len(X) > max_rows else X
    y_hpo = y[:max_rows] if max_rows and len(y) > max_rows else y

    optimal_configs, all_metrics = {}, {}

    for cond in CONDITIONS_TO_RUN:
        log.info(f'\n--- Kondisi {cond} ---')
        t_start = datetime.now()

        # Cek checkpoint dulu — kalau ada, skip HPO
        saved_config, saved_metrics = load_checkpoint(ds_key, cond)
        if saved_config and saved_metrics:
            optimal_configs[cond] = saved_config
            all_metrics[cond]     = saved_metrics
            all_metrics[cond].setdefault('convergence_trace', None)
            log.info(f'  [{cond}] AUC dari checkpoint: {np.mean(saved_metrics["auc"]):.4f}')
            continue

        result      = CONDITION_RUNNERS[cond](X=X_hpo, y=y_hpo, ds_key=ds_key)
        best_config, _, trace = result         
        optimal_configs[cond] = best_config

        metrics = final_evaluation(best_config, X, y, cond, ds_key)
        metrics['convergence_trace'] = trace    
        all_metrics[cond] = metrics

        # Simpan checkpoint segera ke Google Drive
        save_checkpoint(ds_key, cond, best_config,
                        {**{k: [float(v) for v in vals] for k, vals in metrics.items()
                            if k != 'convergence_trace'},
                         'convergence_trace': trace})

        elapsed = (datetime.now() - t_start).seconds // 60
        log.info(f'  [{cond}] Selesai — AUC: {np.mean(metrics["auc"]):.4f} '
                 f'(waktu: {elapsed} menit)')

    log.info('\n--- Uji Statistik ---')
    auc_per_cond = {c: all_metrics[c]['auc'] for c in CONDITIONS_TO_RUN}
    stats_result = run_statistical_tests(auc_per_cond)

    traces_to_plot = {
        c: all_metrics[c]['convergence_trace']
        for c in CONDITIONS_TO_RUN
        if all_metrics[c].get('convergence_trace')
    }
    if traces_to_plot:
        plot_convergence_curves(traces_to_plot, title=f'Konvergensi Proxy-AUC — {ds_key}')

    print(f'\nRINGKASAN HASIL {ds_key}:')
    print(f'{"Kondisi":<10} {"AUC Mean":<12} {"AUC Std":<12} {"F1 Mean":<12}')
    print('-' * 46)
    for cond in CONDITIONS_TO_RUN:
        m = all_metrics[cond]
        print(f'{cond:<10} {np.mean(m["auc"]):<12.4f} {np.std(m["auc"]):<12.4f} {np.mean(m["f1"]):<12.4f}')

    print('\nFriedman rank (lebih rendah = lebih baik):')
    ranks = stats_result.get('mean_friedman_rank', {})
    for c, r in sorted(ranks.items(), key=lambda x: x[1]):
        print(f'  {c}: {r:.3f}')

    output = {
        'dataset': ds_key, 'dataset_name': name,
        'n_samples': len(X), 'n_features': X.shape[1],
        'imbalance_ratio': round(float((y==0).sum()/(y==1).sum()), 2),
        'optimal_configs': optimal_configs,
        'eval_metrics': {c: {k: [round(float(v), 4) for v in vals]
                              for k, vals in m.items()
                              if k != 'convergence_trace' and vals is not None}
                         for c, m in all_metrics.items()},
        'statistics': stats_result,
        'convergence_traces': {c: all_metrics[c].get('convergence_trace')
                                 for c in CONDITIONS_TO_RUN},
    }
    out_path = f'{RESULTS_DIR}/{ds_key}_results.json'
    with open(out_path, 'w') as f:
        json.dump(output, f, indent=2)
    log.info(f'\nHasil disimpan: {out_path}')
    all_results.append(output)

if all_results:
    import shutil
    backup_path = f'{RESULTS_DIR}/backup_{datetime.now():%Y%m%d_%H%M%S}'
    shutil.make_archive(backup_path, 'zip', RESULTS_DIR)
    print(f'\nBackup ZIP: {backup_path}.zip')

print('\nEksperimen selesai!')


2026-06-16 07:05:26,039 - __main__ - INFO - 
2026-06-16 07:05:26,052 - __main__ - INFO - DATASET: K04 — Telco Customer Churn (Telekomunikasi)
2026-06-16 07:05:26,053 - __main__ - INFO - ============================================================


File ditemukan: /kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv

=== HEAD K04 (RAW) ===


2026-06-16 07:05:26,095 - __main__ - INFO - [K04] Memuat: Telco Customer Churn (Telekomunikasi)


Before preprocess: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Total fitur: 21


2026-06-16 07:05:26,129 - __main__ - INFO - [K04] Ukuran awal: (7043, 21)
2026-06-16 07:05:26,136 - __main__ - INFO - [K04] TotalCharges: 11 nilai kosong diisi median=1397.47
2026-06-16 07:05:26,240 - __main__ - INFO - [K04] Ukuran akhir: (7043, 45), IR=2.77
2026-06-16 07:05:26,241 - __main__ - INFO - 
--- Kondisi C1 ---
2026-06-16 07:05:26,243 - __main__ - INFO -   [C1] AUC dari checkpoint: 0.8298
2026-06-16 07:05:26,244 - __main__ - INFO - 
--- Kondisi C2 ---
2026-06-16 07:05:26,247 - __main__ - INFO -   [C2] AUC dari checkpoint: 0.8404
2026-06-16 07:05:26,248 - __main__ - INFO - 
--- Kondisi C3 ---
2026-06-16 07:05:26,250 - __main__ - INFO -   [C3] AUC dari checkpoint: 0.8416
2026-06-16 07:05:26,251 - __main__ - INFO - 
--- Kondisi C4 ---
2026-06-16 07:05:26,254 - __main__ - INFO -   [C4] AUC dari checkpoint: 0.8421
2026-06-16 07:05:26,255 - __main__ - INFO - 
--- Uji Statistik ---


...

2026-06-16 07:05:26,261 - __main__ - INFO -   [STAT] Friedman: chi2=9.2400, p=0.026264 (SIGNIFIKAN)
2026-06-16 07:05:26,606 - __main__ - INFO -   [STAT] C1 vs C2: p_holm=0.0352, |r|=0.822 (large), reject=True
2026-06-16 07:05:26,607 - __main__ - INFO -   [STAT] C1 vs C3: p_holm=0.0488, |r|=0.790 (large), reject=True
2026-06-16 07:05:26,608 - __main__ - INFO -   [STAT] C1 vs C4: p_holm=0.0488, |r|=0.790 (large), reject=True
2026-06-16 07:05:26,609 - __main__ - INFO -   [STAT] C2 vs C3: p_holm=0.4805, |r|=0.403 (medium), reject=False
2026-06-16 07:05:26,610 - __main__ - INFO -   [STAT] C2 vs C4: p_holm=0.4805, |r|=0.467 (medium), reject=False
2026-06-16 07:05:26,611 - __main__ - INFO -   [STAT] C3 vs C4: p_holm=0.4805, |r|=0.371 (medium), reject=False
2026-06-16 07:05:26,709 - __main__ - INFO - 
Hasil disimpan: /kaggle/working/hbp_results2/K04_results.json
2026-06-16 07:05:26,710 - __main__ - INFO - 
2026-06-16 07:05:26,712 - __main__ - INFO - DATASET: K03 — Churn Modelling (Perbankan)
2


RINGKASAN HASIL K04:
Kondisi    AUC Mean     AUC Std      F1 Mean     
----------------------------------------------
C1         0.8298       0.0198       0.7199      
C2         0.8404       0.0191       0.6891      
C3         0.8416       0.0185       0.7150      
C4         0.8421       0.0178       0.7256      

Friedman rank (lebih rendah = lebih baik):
  C4: 1.800
  C3: 2.300
  C2: 2.400
  C1: 3.500
File ditemukan: /kaggle/input/datasets/shrutimechlearn/churn-modelling/Churn_Modelling.csv

=== HEAD K03 (RAW) ===
Before preprocess: ['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
Total fitur: 14
After preprocess shape: (10000, 13)
Feature names after preprocess: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Gender_Female', '

2026-06-16 07:05:27,242 - __main__ - INFO -   [STAT] C1 vs C2: p_holm=0.1367, |r|=0.693 (large), reject=False
2026-06-16 07:05:27,242 - __main__ - INFO -   [STAT] C1 vs C3: p_holm=0.1484, |r|=0.629 (large), reject=False
2026-06-16 07:05:27,243 - __main__ - INFO -   [STAT] C1 vs C4: p_holm=0.0586, |r|=0.790 (large), reject=False
2026-06-16 07:05:27,244 - __main__ - INFO -   [STAT] C2 vs C3: p_holm=0.5566, |r|=0.210 (small), reject=False
2026-06-16 07:05:27,246 - __main__ - INFO -   [STAT] C2 vs C4: p_holm=0.3203, |r|=0.467 (medium), reject=False
2026-06-16 07:05:27,247 - __main__ - INFO -   [STAT] C3 vs C4: p_holm=0.1484, |r|=0.661 (large), reject=False
2026-06-16 07:05:27,314 - __main__ - INFO - 
Hasil disimpan: /kaggle/working/hbp_results2/K03_results.json



RINGKASAN HASIL K03:
Kondisi    AUC Mean     AUC Std      F1 Mean     
----------------------------------------------
C1         0.8535       0.0225       0.7220      
C2         0.8583       0.0199       0.7310      
C3         0.8579       0.0189       0.7272      
C4         0.8598       0.0186       0.7296      

Friedman rank (lebih rendah = lebih baik):
  C4: 1.400
  C2: 2.400
  C3: 2.600
  C1: 3.600


RuntimeError: File size too large, try using force_zip64

In [ ]:
def generate_fold_probs(config, X, y, ds_key, condition):
     skf = StratifiedKFold(n_splits=EVAL_CV_FOLDS, shuffle=True, random_state=SEED)
     folds = []
     for fold_idx, (train_idx, val_idx) in enumerate(tqdm(
         skf.split(X, y), total=EVAL_CV_FOLDS, desc=f'[{condition}] probs')):
         X_tr, X_val = X[train_idx], X[val_idx]
         y_tr, y_val = y[train_idx], y[val_idx]

         # SAMA PERSIS dengan final_evaluation v3.1 (split, scaling, SMOTE, seed)
         X_tr2, X_es, y_tr2, y_es = train_test_split(
             X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)
         scaler = RobustScaler()
         X_tr2 = scaler.fit_transform(X_tr2)
         X_es  = scaler.transform(X_es)
         X_val = scaler.transform(X_val)
         try:
             smt = SMOTETomek(smote=SMOTE(k_neighbors=5, random_state=SEED), random_state=SEED)
             X_tr2, y_tr2 = smt.fit_resample(X_tr2, y_tr2)
         except Exception:
             pass

         model = build_dnn(X_tr2.shape[1], config)
         model.fit(X_tr2, y_tr2, validation_data=(X_es, y_es),
                   batch_size=config['batch_size'], epochs=config['epochs'],
                   callbacks=[tf.keras.callbacks.EarlyStopping(
                       monitor='val_auc', patience=5, min_delta=0.001,
                       restore_best_weights=True, mode='max')],
                   verbose=0)

         prob_es  = model.predict(X_es,  verbose=0).ravel()   # ★ untuk pilih threshold
         prob_val = model.predict(X_val, verbose=0).ravel()   # ★ untuk evaluasi
         folds.append({
             'y_es'    : y_es.astype(int).tolist(),
             'prob_es' : prob_es.astype(float).tolist(),
             'y_val'   : y_val.astype(int).tolist(),
             'prob_val': prob_val.astype(float).tolist(),
         })
         del model; tf.keras.backend.clear_session(); gc.collect()

     path = f'{RESULTS_DIR}/{ds_key}_{condition}_probs.json'
     with open(path, 'w') as f:
         json.dump(folds, f)
     log.info(f'  [{condition}] Probs disimpan: {path}')
     return folds


# DS = 'K04'
DS = 'K03'
Xp, yp, _, _ = load_and_preprocess(DS)
for cond in ['C1', 'C2', 'C3', 'C4']:
     cfg, _ = load_checkpoint(DS, cond)          
     if cfg is None:
         log.warning(f'  [{cond}] Tidak ada checkpoint config — lewati'); continue
     generate_fold_probs({**cfg, 'epochs': FINAL_EPOCHS}, Xp, yp, DS, cond)
print('Selesai regenerasi probs (tanpa PSO/BO).')

2026-06-16 07:13:25,437 - __main__ - INFO - [K03] Memuat: Churn Modelling (Perbankan)
2026-06-16 07:13:25,546 - __main__ - INFO - [K03] Ukuran awal: (10000, 14)
2026-06-16 07:13:25,579 - __main__ - INFO - [K03] Ukuran akhir: (10000, 13), IR=3.91


Checkpoint ditemukan — skip HPO: /kaggle/working/hbp_results2/K03_C1_checkpoint.json


[C1] probs:   0%|          | 0/10 [00:00<?, ?it/s]

2026-06-16 07:13:26.732151: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-06-16 07:13:46.268796: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Checkpoint ditemukan — skip HPO: /kaggle/working/hbp_results2/K03_C2_checkpoint.json


...

2026-06-16 07:20:24.028258: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-06-16 07:20:33.759411: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Checkpoint ditemukan — skip HPO: /kaggle/working/hbp_results2/K03_C4_checkpoint.json


[C4] probs:   0%|          | 0/10 [00:00<?, ?it/s]

2026-06-16 07:23:06.105932: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-06-16 07:23:15.511218: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Selesai regenerasi probs (tanpa PSO/BO).


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
def best_threshold(y_true, prob, grid=None):
     """Pilih threshold yang memaksimalkan macro-F1 pada (y_true, prob)."""
     if grid is None:
         grid = np.linspace(0.05, 0.95, 91)
     best_t, best_f1 = 0.5, -1.0
     for t in grid:
         f1 = f1_score(y_true, (prob >= t).astype(int), average='macro', zero_division=0)
         if f1 > best_f1:
             best_f1, best_t = f1, t
     return best_t

def evaluate_threshold_strategies(ds_key, conditions=('C1','C2','C3','C4')):
     print(f'{"Kondisi":<8} {"F1@0.5":>9} {"F1@opt*":>9} {"Δ":>7} {"thr* (rata2)":>13}')
     print('-' * 52)
     summary = {}
     for cond in conditions:
        path = f'{RESULTS_DIR}/{ds_key}_{cond}_probs.json'
        if not os.path.exists(path):
             print(f'{cond:<8} (probs belum ada)'); continue
        with open(path) as f:
             folds = json.load(f)

        f1_05, f1_opt, thrs = [], [], []
        for fold in folds:
             y_es   = np.array(fold['y_es']);   p_es  = np.array(fold['prob_es'])
             y_val  = np.array(fold['y_val']);  p_val = np.array(fold['prob_val'])
             t = best_threshold(y_es, p_es)
             thrs.append(t)
             f1_05.append(f1_score(y_val, (p_val >= 0.5).astype(int), average='macro', zero_division=0))
             f1_opt.append(f1_score(y_val, (p_val >= t).astype(int), average='macro', zero_division=0))

        m05, mopt = np.mean(f1_05), np.mean(f1_opt)
        print(f'{cond:<8} {m05:>9.4f} {mopt:>9.4f} {mopt-m05:>+7.4f} {np.mean(thrs):>13.3f}')
        summary[cond] = {'f1@0.5': round(m05,4), 'f1@opt': round(mopt,4),
                         'delta': round(mopt-m05,4), 'thr_mean': round(float(np.mean(thrs)),3),
                          'thr_per_fold': [round(t,3) for t in thrs]}
     return summary

# DS = 'K04'
DS = 'K03'
print(f'=== THRESHOLD TUNING (macro-F1) — {DS} ===')
print('(threshold dipilih pada validation internal, diterapkan ke fold test)\n')
result = evaluate_threshold_strategies(DS)

out = f'{RESULTS_DIR}/{DS}_threshold_tuning.json'
with open(out, 'w') as f:
     json.dump(result, f, indent=2)
print(f'\nDisimpan: {out}')


=== THRESHOLD TUNING (macro-F1) — K03 ===
(threshold dipilih pada validation internal, diterapkan ke fold test)

Kondisi     F1@0.5   F1@opt*       Δ  thr* (rata2)
----------------------------------------------------
C1          0.7224    0.7528 +0.0304         0.688
C2          0.7310    0.7578 +0.0268         0.686
C3          0.7270    0.7561 +0.0292         0.681
C4          0.7292    0.7581 +0.0289         0.676

Disimpan: /kaggle/working/hbp_results2/K03_threshold_tuning.json


In [ ]:
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                             balanced_accuracy_score)

def _best_threshold(y_true, prob, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_true, (prob >= t).astype(int), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t

def compute_full_metrics(ds_key, conditions=('C1', 'C2', 'C3', 'C4')):
    rows = {}
    for cond in conditions:
        path = f'{RESULTS_DIR}/{ds_key}_{cond}_probs.json'
        if not os.path.exists(path):
            print(f'  [{cond}] probs tidak ada di {path} — lewati')
            continue
        with open(path) as f:
            folds = json.load(f)

        acc = {k: [] for k in
               ['auc', 'aucpr', 'f1_05', 'f1_opt', 'f1_min_opt', 'bal_acc_opt', 'thr']}
        for fold in folds:
            y_es  = np.array(fold['y_es']);  p_es  = np.array(fold['prob_es'])
            y_val = np.array(fold['y_val']); p_val = np.array(fold['prob_val'])

            t = _best_threshold(y_es, p_es)                
            pred_05  = (p_val >= 0.5).astype(int)
            pred_opt = (p_val >= t).astype(int)

            acc['auc'].append(roc_auc_score(y_val, p_val))
            acc['aucpr'].append(average_precision_score(y_val, p_val))
            acc['f1_05'].append(f1_score(y_val, pred_05,  average='macro',  zero_division=0))
            acc['f1_opt'].append(f1_score(y_val, pred_opt, average='macro',  zero_division=0))
            acc['f1_min_opt'].append(f1_score(y_val, pred_opt, pos_label=1, average='binary', zero_division=0))
            acc['bal_acc_opt'].append(balanced_accuracy_score(y_val, pred_opt))
            acc['thr'].append(t)
        rows[cond] = {k: float(np.mean(v)) for k, v in acc.items()}
        rows[cond]['auc_std'] = float(np.std(acc['auc']))
    return rows

def print_table(ds_key, rows):
    print(f'\n=== METRIK LENGKAP — {ds_key} '
          f'(threshold opt: dipilih di val-internal, diterapkan ke test) ===')
    hdr = (f'{"Kondisi":<8}{"AUC":>9}{"AUC-PR":>9}{"F1@0.5":>9}'
           f'{"F1@opt":>9}{"F1-min":>9}{"BalAcc":>9}{"thr*":>7}')
    print(hdr); print('-' * len(hdr))
    for c in ['C1', 'C2', 'C3', 'C4']:
        if c not in rows: continue
        r = rows[c]
        print(f'{c:<8}{r["auc"]:>9.4f}{r["aucpr"]:>9.4f}{r["f1_05"]:>9.4f}'
              f'{r["f1_opt"]:>9.4f}{r["f1_min_opt"]:>9.4f}{r["bal_acc_opt"]:>9.4f}{r["thr"]:>7.3f}')

all_full = {}
for DS in ['K04', 'K03']:
    rows = compute_full_metrics(DS)
    if rows:
        print_table(DS, rows)
        all_full[DS] = rows
        with open(f'{RESULTS_DIR}/{DS}_full_metrics.json', 'w') as f:
            json.dump(rows, f, indent=2)

print('\nCatatan: AUC & AUC-PR tidak bergantung threshold (identik utk @0.5 & @opt).')
print('F1-min = F1 kelas churn(=1). BalAcc = (sensitivity+specificity)/2.')
print('Tersimpan: {DS}_full_metrics.json untuk K04 & K03.')


=== METRIK LENGKAP — K04 (threshold opt: dipilih di val-internal, diterapkan ke test) ===
Kondisi       AUC   AUC-PR   F1@0.5   F1@opt   F1-min   BalAcc   thr*
---------------------------------------------------------------------
C1         0.8343   0.6393   0.7143   0.7348   0.6151   0.7397  0.675
C2         0.8404   0.6539   0.6891   0.7337   0.6145   0.7407  0.610
C3         0.8416   0.6536   0.7150   0.7306   0.6076   0.7353  0.613
C4         0.8421   0.6512   0.7256   0.7363   0.6160   0.7401  0.582

=== METRIK LENGKAP — K03 (threshold opt: dipilih di val-internal, diterapkan ke test) ===
Kondisi       AUC   AUC-PR   F1@0.5   F1@opt   F1-min   BalAcc   thr*
---------------------------------------------------------------------
C1         0.8535   0.6789   0.7224   0.7528   0.6005   0.7425  0.688
C2         0.8583   0.6944   0.7310   0.7578   0.6072   0.7456  0.686
C3         0.8579   0.6911   0.7270   0.7561   0.6068   0.7482  0.681
C4         0.8598   0.6949   0.7292   0.7581   0